<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/Step02dSam2TrackFull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 02d · Track with SAM 2.1: full hour

The tracker for the pipeline: **SAM 2.1** follows the cow through the **whole clip** (replaces ByteTrack, Step 02b). Run Step 02c first if you changed settings.

**How:** YOLO26n finds the cow at the start of each 2-minute chunk, SAM 2.1 Small follows her mask through the chunk, and chunks are joined where the boxes overlap. Each SAM box is checked against YOLO's: **Agrees** (`ok`), **NoYoloBox** (`sam`: YOLO lost her, SAM kept her), **Disagrees** (`??`: SAM may have drifted; check the preview).

**Takes about 40–60 min on a T4** (31 chunks). **If it stops** (disconnect, GPU time used up), open the notebook again, even on someone else's account, and run cells 1–3 with the **same settings**: finished chunks are kept in `Chunks/` on the Shared Drive and it continues from the next one. Changing a setting starts over.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step02dSam2Full/`:

- `Tracks.csv`: **input for Step 03b**; `TrackSummary.csv`
- `Comparison.csv`: ByteTrack (02b) vs SAM 2 over the whole clip
- `PreviewTracks.mp4`: masks + boxes + IDs, 4x speed (open it from Drive)
- `Chunks/`: per-chunk pieces, needed to resume

Needs a **T4 GPU**. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install YOLO/SAM.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Sam2Track import Sam2Track
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem
ClipPath = VideoOutputDir / "ReferenceClip.mp4"                       # step 01
CompareTracksPath = VideoOutputDir / "Step02bFull" / "Tracks.csv"     # ByteTrack result to compare with (optional)
OutputDir = VideoOutputDir / "Step02dSam2Full"

StartSeconds = 0    # seconds into the clip (add 03:35:11 for video time)
EndSeconds = None   # None = to the end of the clip
TargetFps = 2          # the camera only updates ~2 times a second, so more frames add little
ChunkSeconds = 120     # SAM 2 memory grows per frame; restart it from a YOLO box every 2 min
SeedConfidence = 0.4   # YOLO box needed to start following a cow
Preview = True         # False = no preview video (faster)
PreviewSpeed = 4
Resume = True          # True = continue a stopped run (same settings); False = start over

assert ClipPath.exists(), f"Run step 01 first: {ClipPath}"

In [ ]:
# 3. Run: track with SAM 2 (continues where a stopped run ended), compare with ByteTrack, show the preview.
import pandas as pd
from IPython.display import Video, display

Stats = Sam2Track(ClipPath, OutputDir, StartSeconds, EndSeconds, TargetFps=TargetFps, ChunkSeconds=ChunkSeconds,
                  SeedConfidence=SeedConfidence, CompareTracksPath=CompareTracksPath,
                  Preview=Preview, PreviewSpeed=PreviewSpeed, Resume=Resume)

display(pd.read_csv(OutputDir / "Comparison.csv"))
display(pd.read_csv(OutputDir / "TrackSummary.csv"))
PreviewPath = OutputDir / "PreviewTracks.mp4"
if PreviewPath.exists():
    PreviewMb = PreviewPath.stat().st_size / 2**20
    print(f"Preview: {PreviewPath} ({PreviewMb:.0f} MB)")
    if PreviewMb < 50:   # bigger files are too heavy to play inside the notebook; open them from Drive
        display(Video(str(PreviewPath), embed=True, width=960))

## About this step

**Models:** **YOLO26n** (detection, only to start each chunk and to check) + **SAM 2.1 Small in video mode** (Ultralytics `SAM2VideoPredictor`, `sam2.1_s.pt`) for following the cow.

**Why SAM instead of ByteTrack:** ByteTrack links boxes by position and gives a new TrackId whenever YOLO loses her for > 10 s, so one cow became 47 tracks. SAM follows her outline by appearance: on 1080–1980 s it kept her as one track with 97 % YOLO agreement (Step 02c).

**Still true:** a TrackId is not a cow. A new track starts when a chunk cannot be linked to the previous one (she moved a lot at the border, or no confident YOLO box at a chunk start), and a calf can get its own track once it is big enough for YOLO.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.